# 03 — Topic modelling (LDA)

Cluster the residual (non-government, non-candidate) ad bodies into latent topics so we can:

- Filter out commercial/spam topics (fashion, fitness apps, retail) from the political-adjacent subset.
- Surface what *kinds* of political messaging are circulating outside party/candidate channels — climate, cost-of-living, Voice, housing, etc.
- Layer topic labels into the v3 parquet for cross-tabbing with sentiment (notebook 04) and spend/impressions.

Approach: fit a single LDA model on the full residual corpus, eyeball the top words per topic, hand-label each topic in a CSV, join the labels back to the corpus.

## 1. Preprocessing

Load v2 parquet, filter to one row per ad (`ad_seq_no = 1`) and non-classified (`match_type IS NULL`). Extract the first creative body, tokenise with `RegexTokenizer`, drop stop words (English defaults + domain-specific noise). Vectorise word counts with `CountVectorizer` — raw counts, not TF-IDF, since LDA expects integer term frequencies.

Cache the vectorised features so subsequent LDA fits (at different `k`) don't re-run preprocessing.

### 1.1 Spark session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, coalesce, concat_ws, expr, length, lit

# Same parquet-committer override as notebook 02 — cluster default points at an EMR
# class whose JAR isn't on the classpath.
spark = SparkSession.builder \
    .appName('FB_API_topics') \
    .config('spark.sql.parquet.output.committer.class',
            'org.apache.parquet.hadoop.ParquetOutputCommitter') \
    .config('mapreduce.fileoutputcommitter.algorithm.version', '2') \
    .getOrCreate()

print('Master:', spark.sparkContext.master)
print('Spark version:', spark.version)

### 1.2 Paths

In [ ]:
V2_PATH           = '/user/s3348393/main/preprocessing/v2/parquet'
INTERMEDIATE_PATH = '/user/s3348393/main/preprocessing/v3/intermediate_parquet'  # corpus + topic_id
V3_PATH           = '/user/s3348393/main/preprocessing/v3/parquet'               # final with labels

TOPIC_TERMS_CSV   = '../data/topic_terms.csv'    # output: top words per topic (for human review)
TOPIC_LABELS_CSV  = '../data/topic_labels.csv'   # input: human-edited labels

### 1.3 Load v2 and filter to the residual corpus

Keep one row per ad (`ad_seq_no = 1`) and only ads that didn't classify as candidate/party/government.

Body-text extraction is more inclusive than just `creative_bodies[0]`:

1. `first_non_empty(arr)` returns the **first non-null, non-empty** element of an array column — recovers ads where the first creative-body entry is null but a later one is real (multi-variant ads), or where the singular `ad_creative_body` was null but other text fields are populated.
2. Concatenate `creative_bodies` + `creative_link_descs` + `creative_link_titles` into a single document — more text = better LDA signal, and ads with no body but a populated link description ("Sign the petition", "Donate now") still contribute.
3. Drop ads with no text in any of these fields — LDA can't process empty documents.

`creative_link_captions` is skipped because it's usually just a domain name (low signal, noise).

In [ ]:
df = spark.read.parquet(V2_PATH)
print('All v2 rows:    ', df.count())


def first_non_empty(col_name):
    """First non-null, non-empty element of an array column. Returns null if none."""
    return expr(f"filter({col_name}, x -> x is not null and length(x) > 0)[0]")


corpus = df.filter((col('ad_seq_no') == 1) & col('match_type').isNull()) \
    .withColumn('body_text',  first_non_empty('creative_bodies')) \
    .withColumn('desc_text',  first_non_empty('creative_link_descs')) \
    .withColumn('title_text', first_non_empty('creative_link_titles')) \
    .withColumn('body',
        concat_ws(' ',
            coalesce(col('body_text'),  lit('')),
            coalesce(col('desc_text'),  lit('')),
            coalesce(col('title_text'), lit('')),
        )
    ) \
    .filter(length(col('body')) > 0) \
    .drop('body_text', 'desc_text', 'title_text')

print('Residual corpus:', corpus.count())
corpus.select('page_name', 'bylines', 'body').show(3, truncate=80)

### 1.4 Stop words

English defaults plus a small list of domain-specific noise: URL fragments, generic call-to-action words. Keep this list deliberately short — `minDF` and `maxDF` in `CountVectorizer` will handle most of the frequency-based filtering automatically. Iterate after the first LDA fit if specific tokens are dominating topics with no signal.

In [ ]:
from pyspark.ml.feature import StopWordsRemover

stop_words = StopWordsRemover.loadDefaultStopWords('english') + [
    # URL / web junk that survives tokenisation
    'https', 'http', 'www', 'com', 'org', 'au', 'co', 'html',
    # generic CTA noise
    'click', 'learn', 'sign', 'today', 'also', 'will', 'can', 'get',
    'see', 'know', 'one', 'two', 'new', 'now', 'us',
]

print('Stop-words list size:', len(stop_words))

### 1.5 Preprocessing pipeline

Three stages: `RegexTokenizer` (split on `\W+`, lowercase, drop tokens shorter than 2 chars) → `StopWordsRemover` (English defaults + domain noise) → `CountVectorizer` (vocab≤5,000, term must appear in ≥50 ads, term must appear in ≤30% of ads).

Wrap in a `Pipeline` so we can `.fit().transform()` in one go and have a single fitted artefact to introspect afterwards.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, CountVectorizer

tokenizer = RegexTokenizer(
    inputCol='body', outputCol='raw_tokens',
    pattern=r'\W+', toLowercase=True, minTokenLength=2,
)

remover = StopWordsRemover(
    inputCol='raw_tokens', outputCol='tokens',
    stopWords=stop_words,
)

vectorizer = CountVectorizer(
    inputCol='tokens', outputCol='features',
    vocabSize=5000,
    minDF=50,     # term must appear in ≥50 ads to be kept
    maxDF=0.3,    # term appearing in >30% of ads is dropped (auto stop-word filter)
)

prep_pipeline = Pipeline(stages=[tokenizer, remover, vectorizer])

### 1.6 Fit, transform, cache

Fit the pipeline once. The output `features_df` carries every original column plus `raw_tokens`, `tokens`, and `features` (the sparse count vector LDA consumes). Cache it so the k-sweep in section 2 doesn't re-run preprocessing per `k`.

The `.count()` call forces Spark to actually materialise the cache — without it, the cache is registered lazily and nothing happens until something else triggers an action.

In [ ]:
prep_model  = prep_pipeline.fit(corpus)
features_df = prep_model.transform(corpus).cache()

print('Cached rows:    ', features_df.count())

vocab = prep_model.stages[-1].vocabulary
print('Vocabulary size:', len(vocab))
print('\nTop 30 vocabulary terms (most frequent first):')
print(vocab[:30])

## 2. Explore `k` on a sample

Fit LDA at several `k` values (5, 10, 15, 20) on a 10% sample of the cached features. Print top-12 words per topic for each `k`. Eyeball the printouts to pick a `k` where topics are distinct and each list reads as a coherent theme.

Fix `seed=42` so comparing `k=10` vs `k=15` isn't muddled by random init differences. Cheap and disposable — no parquet writes from this section.

In [ ]:
from pyspark.ml.clustering import LDA

# 10% sample of the cached features. Cache the sample too — the four LDA fits
# below will scan it repeatedly. seed=42 keeps the sample composition stable.
sample_df = features_df.sample(0.1, seed=42).cache()
print(f'Sample size: {sample_df.count():,}')

# Lookup from CountVectorizer integer term indices back to readable words.
vocab = prep_model.stages[-1].vocabulary

# Fit LDA at each k. seed=42 fixed so k=5 vs k=10 etc. are comparable.
for k in [5, 10, 15, 20]:
    print(f'\n=== k = {k} ===')
    lda = LDA(featuresCol='features', k=k, maxIter=20, seed=42)
    model = lda.fit(sample_df)
    topics = model.describeTopics(maxTermsPerTopic=12).collect()
    for row in topics:
        words = ' '.join(vocab[i] for i in row.termIndices)
        print(f'  Topic {row.topic:>2}: {words}')

sample_df.unpersist()

## 3. Final fit on full corpus

Refit LDA at the chosen `k` on the full cached features. Transform the corpus to attach `topicDistribution` (length-`k` vector) and `topic_id` (argmax) to every ad. Persist:

- **Intermediate parquet** — corpus + `topicDistribution` + `topic_id`. Expensive to recompute, so write it once.
- **`data/topic_terms.csv`** — small file, one row per topic, top-15 most representative terms each.

Optionally save the fitted `PipelineModel` so the same topic model can be applied to new ads later.

## 4. Manual labelling (out-of-notebook)

Open `data/topic_terms.csv` in a spreadsheet, add a `label` column with human-readable names (e.g. `climate`, `cost_of_living`, `voice_referendum`, `commercial_retail`, `noise`). Save as `data/topic_labels.csv`.

Topics that look like commercial noise (fashion brands, fitness app keywords, etc.) get labels like `commercial_*` or `noise` — these are the categories notebook 04 / later filters will drop.

## 5. Join labels back

Read intermediate parquet + `data/topic_labels.csv`. Broadcast-join on `topic_id` to add a `topic_label` column. Write the result as v3 parquet — same schema as v2 with `topicDistribution`, `topic_id`, and `topic_label` appended.

Fully re-runnable: tweak labels, re-run this section, no LDA refit needed.